# ⚡ ZAC 2026 LifeMate — KIẾN TRÚC HAI GIAI ĐOẠN (V2 TWO-STAGE PIPELINE)
### Pipeline Hai Giai Đoạn Đột Phá — Tối Ưu Điểm Số & Cắt Giảm 50% Thời Gian Suy Luận trên Kaggle GPU (T4 / P100)

**Tại sao cần Version 2 (Two-Stage)?**
- **Vấn đề của V1 (Single-Stage):** Toàn bộ ~140 công cụ thuộc 46 ứng dụng phải nhồi nhét vào prompt khiến ngữ cảnh dài ~3,500+ tokens. Mô hình Qwen 2.5 3B dễ bị quá tải, nhầm lẫn công cụ giữa các miền và giải mã chậm (15s - 25s/mẫu), dễ chạm trần timeout 30s.
- **Đột phá của V2 (Two-Stage Architecture):**
  1. **Giai đoạn 1 (Stage 1 Router & Context Condenser):** Đọc toàn bộ audio transcript (lịch sử + câu hiện tại) kết hợp với danh sách ứng dụng đã cài (installed_apps chỉ gồm tên miền & mô tả 1 dòng). Trích xuất danh sách app cần dùng (`selected_apps`) và cô đọng ngữ cảnh lịch sử (`condensed_history`). Prompt chỉ ~300 tokens, xử lý trong ~0.7s.
  2. **Mạng lưới Đồng hành (Companion Network):** Áp dụng nguyên tắc an toàn *"Có thể lấy thừa nhưng tuyệt đối không được thiếu"* (ví dụ: thanh toán hoá đơn thì tự động bổ sung ví điện tử và ngân hàng; đặt xe bổ sung bản đồ).
  3. **Giai đoạn 2 (Targeted ReAct Planner):** Chỉ nạp vào prompt các công cụ thuộc các app được chọn ở Giai đoạn 1 (giảm từ 140 tools xuống còn 4-10 tools). LLM tập trung 100% vào miền nghiệp vụ, không ảo giác, tốc độ giải mã tăng gấp 2 lần!

### Bước 1: Cài đặt thư viện phụ thuộc

In [ ]:
!pip install -q faster-whisper soundfile python-dotenv transformers accelerate rank-bm25 pydantic openai

### Bước 2: Thiết lập môi trường và tự động nhận diện dữ liệu trên Kaggle

In [ ]:
import os
import sys
import json
import time
import zipfile
from pathlib import Path
from tqdm import tqdm

# Đảm bảo nạp các module từ repository
REPO_DIR = Path("/kaggle/working/zac2026-lifemate")
if not REPO_DIR.exists():
    REPO_DIR = Path.cwd()
sys.path.insert(0, str(REPO_DIR))

# Tự động nhận diện thư mục dữ liệu train+public_test
DATA_CANDIDATES = [
    Path("/kaggle/input/datasets/n107hoangtuong/zalo-competion/train+public_test"),
    Path("/kaggle/input/zalo-ai-challenge-2026/train+public_test"),
    Path("/kaggle/input/train+public_test"),
]
DATA_ROOT = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_ROOT is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "public_test.jsonl" in files:
            DATA_ROOT = Path(root).parent
            break

print(f"✅ Đã nhận diện DATA_ROOT tại: {DATA_ROOT}")
OUTPUT_DIR = Path("/kaggle/working/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Bước 3: Gỡ băng âm thanh ASR siêu tốc (PhoWhisper CTranslate2 trên GPU CUDA)

In [ ]:
import gc
import torch
from faster_whisper import WhisperModel
from src.asr.transcribe import load_audio_file
from src.asr.post_processor import LexiconPostProcessor

MODEL_ID = "nunu-willump/PhoWhisper-medium-ct2"
print(f"⏳ Đang nạp mô hình ASR: {MODEL_ID} trên CUDA (float16)...")
asr_model = WhisperModel(MODEL_ID, device="cuda", compute_type="float16")
post_proc = LexiconPostProcessor()

SPLIT = "public_test"
INPUT_FILE = DATA_ROOT / SPLIT / f"{SPLIT}.jsonl"
AUDIO_DIR = DATA_ROOT / SPLIT
TRANSCRIPT_OUT = OUTPUT_DIR / f"{SPLIT}_transcripts.jsonl"

records = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"🎙️ Bắt đầu gỡ băng FULL tập {SPLIT} ({len(records)} mẫu)...")
with open(TRANSCRIPT_OUT, "w", encoding="utf-8") as out_f:
    for item in tqdm(records, desc=f"Transcribing {SPLIT}"):
        sample_id = item["id"]
        audio_path = AUDIO_DIR / item["audio"]
        audio_data = load_audio_file(audio_path, target_sr=16000)
        segments, _ = asr_model.transcribe(audio_data, language="vi", beam_size=1, temperature=0.0)
        raw_text = " ".join([s.text.strip() for s in segments]).strip()
        final_text = post_proc.process(raw_text)
        
        hist_transcripts = []
        if "history" in item and isinstance(item["history"], list):
            for h_idx, turn in enumerate(item["history"]):
                if turn.get("audio"):
                    h_path = AUDIO_DIR / turn["audio"]
                    if h_path.exists():
                        h_data = load_audio_file(h_path, target_sr=16000)
                        h_segs, _ = asr_model.transcribe(h_data, language="vi", beam_size=1, temperature=0.0)
                        h_raw = " ".join([s.text.strip() for s in h_segs]).strip()
                        hist_transcripts.append({"history_index": h_idx, "transcript": post_proc.process(h_raw)})
        
        res = {"id": sample_id, "transcript": final_text, "history_user_transcripts": hist_transcripts}
        out_f.write(json.dumps(res, ensure_ascii=False) + "\n")

print(f"✅ Đã xuất bản gỡ băng thành công: {TRANSCRIPT_OUT}")
# Giải phóng VRAM để sẵn sàng chạy LLM / Agent
del asr_model
gc.collect()
torch.cuda.empty_cache()

### Bước 4: Chạy Suy Luận V2 (Hai Giai Đoạn) & Tự Động Đóng Gói Submission
Điểm cốt lõi:
- Sử dụng `--workers 2` kết hợp `max_tokens=256` để tối ưu cho GPU T4.
- Chạy `TwoStageReActAgent`: Stage 1 chọn app & tóm tắt history, Stage 2 thực thi với số tools tối giản.
- Kiểm định tự động chuẩn quy chế BTC bằng `schema_validate`.
- Xuất `/kaggle/working/submission.zip` để nộp trực tiếp.

In [ ]:
# Thay đổi URL ngrok LLM nếu bạn host riêng, hoặc để mặc định cấu hình từ .env
NGROK_LLM_URL = "https://b817-136-85-118-33.ngrok-free.app/v1"

!python scripts/run_v2_and_evaluate.py \
    --data_root "{DATA_ROOT}" \
    --split public_test \
    --asr_file "{TRANSCRIPT_OUT}" \
    --workers 2 \
    --max_tokens 256 \
    --time_budget 45.0 \
    --llm_endpoint "{NGROK_LLM_URL}" \
    --output "/kaggle/working/submission_v2.jsonl" \
    --zip_output "/kaggle/working/submission_v2.zip"

print("🎉 ĐÃ HOÀN TẤT SUY LUẬN V2 TRÊN TOÀN BỘ PUBLIC TEST!")
print("File nộp bài chuẩn BTC đã sẵn sàng: /kaggle/working/submission_v2.zip")

### Bước 5 (Tùy chọn): So sánh Đối đầu V1 vs V2 trên 20 mẫu Benchmark có nhãn
Chạy đối chiếu trực tiếp trên `data/val_benchmark_200.jsonl` để kiểm tra độ trễ (latency), số tools nạp vào prompt và điểm số BTC Gold giữa 2 phiên bản.

In [ ]:
!python scripts/compare_v1_v2.py \
    --num_samples 20 \
    --max_tokens 256 \
    --time_budget 30.0 \
    --llm_endpoint "{NGROK_LLM_URL}"

# Hiển thị báo cáo Markdown so sánh
from IPython.display import display, Markdown
if Path("reports/comparison_v1_v2.md").exists():
    with open("reports/comparison_v1_v2.md", "r", encoding="utf-8") as f:
        display(Markdown(f.read()))